# Cross-Cohort Niche Label Transfer: H&E-Supervised Spatial Tissue Analysis

## Abstract

Spatial transcriptomics (e.g., Xenium) provides rich biological context but is cost-prohibitive at scale.
H&E staining is universally available. This notebook demonstrates a four-step pipeline that uses
Xenium data only during label construction, then trains a model that predicts biologically meaningful
niche types from H&E-derived cell-neighbor composition features alone.

The cohort consists of 4 biological samples (S1–S4), each sectioned at three tissue levels
(Top / Mid / Bot), yielding **12 tissue pieces** labeled P1–P12. Two independent groups are formed:

- **Group 1 (G1)**: P1–P6 (samples S1, S2)
- **Group 2 (G2)**: P7–P12 (samples S3, S4)

WSInsight niche discovery is run **independently** on each group, so niche IDs are not shared across groups.

## Core Scientific Contribution

1. **Independent niche discovery** per cohort group prevents label leakage between groups.
2. **Within-group label consistency** (G1A vs G1B): GOEA+LLM labeling is run independently on
   two halves of G1 that share the same niche ID space. Agreement in human-readable labels for the
   same niche ID confirms that cell-neighbor composition reliably encodes biological niche identity.
3. **Cross-group label transfer** (G1 → G2): A neural network trained on G1 H&E features predicts
   niche types on G2. G2's independently derived GOEA+LLM labels serve as held-out ground truth.

## Problem Statement

Let $\phi_i \in \mathbb{R}^{2d}$ be the neighbor-composition feature vector for niche group $i$
(mean and standard deviation of $d$ H&E-derived features over all member cells):
$$
\phi_i = [\mu_{i,1}, \dots, \mu_{i,d},\ \sigma_{i,1}, \dots, \sigma_{i,d}]
$$

Learn a function $f_\theta(\phi_i) \rightarrow y_i$ where $y_i$ is a human-readable niche type
(e.g., *tumor proliferative*, *immune interferon*, *TLS lymphoid*).

At inference time, Xenium is **not required**.

---

## Pipeline Overview

| Step | Input | Method | Output | Validation |
|------|-------|--------|--------|------------|
| 0 | P1–P12 H&E | WSInsight (2 independent runs) | G1 niche IDs, G2 niche IDs | — |
| 1 (G1A) | G1A Xenium (P1–P3) | DE → GOEA → LLM | G1 niche labels | — |
| 2 (G1B) | G1B Xenium (P4–P6) | Independent DE → GOEA → LLM | G1B niche labels | AUC/CM vs G1A labels |
| 3 | G1 H&E features + labels | Neural network | Trained classifier | G1B internal validation |
| 4 | G2 H&E features | Trained NN (no Xenium) | Predicted G2 labels | G2 Xenium GOEA+LLM labels |

## Stage 0 · Setup

### 0-A: Install Dependencies

Install all required Python packages. Run once per new environment; safe to re-run (pip is idempotent).

In [ ]:
# Install required packages.
# -q = quiet mode (suppresses most pip output).
%pip install -q goatools mygene openai python-dotenv scanpy seaborn scikit-learn torch tqdm

### 0-B: Imports

Load all standard libraries used throughout the notebook.

In [ ]:
# Standard and scientific imports used throughout every stage.
import os
import re
import ast
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import mygene
from tqdm.auto import tqdm

from goatools.base import download_go_basic_obo, download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_curve, auc,
)

# Scanpy display settings: less verbose, clean figures.
sc.settings.verbosity = 1
sc.set_figure_params(dpi=110, frameon=False)

# MyGene client for symbol → Entrez ID mapping used in GOEA.
mg = mygene.MyGeneInfo()

print("All imports successful.")

### 0-C: Configuration

All paths, sample names, and hyperparameters live here. **Edit only this cell** when adapting to a new dataset.

Sample assignment:
```
G1A  P1-P3  S1_Top, S1_Mid, S1_Bot
G1B  P4-P6  S2_Top, S2_Mid, S2_Bot
G2   P7-P12 S3_Top, S3_Mid, S3_Bot, S4_Top, S4_Mid, S4_Bot
```

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SAMPLE DEFINITIONS
# Adjust only the strings below when working with a different cohort.
# ──────────────────────────────────────────────────────────────────────────────

# G1A: first half of Group 1 (used to build niche labels via GOEA+LLM)
G1A_STEMS = [
    "Human_Breast_Biomarkers_S1_Top_he_image.ome",
    "Human_Breast_Biomarkers_S1_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S1_Bot_he_image.ome",
]

# G1B: second half of Group 1 (independent GOEA+LLM for label consistency check)
G1B_STEMS = [
    "Human_Breast_Biomarkers_S2_Top_he_image.ome",
    "Human_Breast_Biomarkers_S2_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S2_Bot_he_image.ome",
]

# G2: second independent group (cross-cohort transfer target)
G2_STEMS = [
    "Human_Breast_Biomarkers_S3_Top_he_image.ome",
    "Human_Breast_Biomarkers_S3_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S3_Bot_he_image.ome",
    "Human_Breast_Biomarkers_S4_Top_he_image.ome",
    "Human_Breast_Biomarkers_S4_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S4_Bot_he_image.ome",
]

# Derived convenience lists.
G1_STEMS = G1A_STEMS + G1B_STEMS  # all Group 1 samples
ALL_STEMS = G1_STEMS + G2_STEMS   # full cohort

# ──────────────────────────────────────────────────────────────────────────────
# PATHS
# ──────────────────────────────────────────────────────────────────────────────
OUTPUTS_VERSION = "v3"  # change to "v4" etc. when re-running with new wsinsight outputs
OUTPUTS_ROOT = Path("outputs") / OUTPUTS_VERSION
IMPORTED_XENIUM_DIR = OUTPUTS_ROOT / "imported-xenium"

# ──────────────────────────────────────────────────────────────────────────────
# CENTRAL CONFIG DICT  (carry-forward variable used by all downstream cells)
# ──────────────────────────────────────────────────────────────────────────────
CFG = {
    # Data sources
    "slides_dir":         Path("data"),
    "image_list_txt":     Path("data/imagelist.txt"),
    "sptx_map_tsv":       Path("data/sptxlist.tsv"),
    "run_script":         Path("run-wsinsight-all.sh"),

    # Output roots
    "outputs_version":    OUTPUTS_VERSION,
    "outputs_root":       OUTPUTS_ROOT,
    "imported_xenium_dir": IMPORTED_XENIUM_DIR,

    # Sub-output directories (one per stage)
    "outdir_g1a":         OUTPUTS_ROOT / "v2_g1a_goea",   # G1A GOEA+LLM outputs
    "outdir_g1b":         OUTPUTS_ROOT / "v2_g1b_goea",   # G1B GOEA+LLM outputs
    "outdir_g2":          OUTPUTS_ROOT / "v2_g2_goea",    # G2  GOEA+LLM outputs
    "nn_dir":             OUTPUTS_ROOT / "v2_nn",          # NN training/evaluation

    # Sample lists
    "g1a_stems":          G1A_STEMS,
    "g1b_stems":          G1B_STEMS,
    "g1_stems":           G1_STEMS,
    "g2_stems":           G2_STEMS,
    "all_stems":          ALL_STEMS,

    # Reproducibility seed
    "seed": 42,

    # Neural network hyperparameters (adjust freely)
    "nn_epochs":          50,
    "nn_batch_size":      64,
    "nn_lr":              1e-3,
    "nn_weight_decay":    1e-4,
    "nn_hidden":          [256, 128],   # hidden layer sizes
    "nn_dropout":         0.3,

    # DE / GOEA thresholds (adjust freely)
    "de_min_cells":       20,           # minimum in-group cells for DE test
    "de_pval":            0.05,         # adjusted p-value threshold
    "de_logfc":           0.25,         # log2 fold-change threshold
    "goea_fdr":           0.05,         # FDR threshold for significant GO terms
    "goea_min_genes":     10,           # minimum mapped study genes to run GOEA
    "llm_top_terms":      8,            # number of GO terms per LLM prompt
}

# Create all output directories now so later saves never fail.
for _key in ["outdir_g1a", "outdir_g1b", "outdir_g2", "nn_dir"]:
    CFG[_key].mkdir(parents=True, exist_ok=True)

# Set global random seeds for reproducibility.
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

# Human-readable summary.
print("Configuration loaded.")
print(f"  Outputs version : {CFG['outputs_version']}")
print(f"  All samples     : {len(ALL_STEMS)}  (G1A={len(G1A_STEMS)}, G1B={len(G1B_STEMS)}, G2={len(G2_STEMS)})")
print(f"  G1A stems       : {G1A_STEMS}")
print(f"  G1B stems       : {G1B_STEMS}")
print(f"  G2  stems       : {G2_STEMS}")

---

## Stage 0-D · WSInsight Pipeline

### Background (beginner)

**Why two independent runs?**
If G1 and G2 were clustered together, their niche IDs would be entangled. Running independently
ensures that a niche ID in G2 is defined purely from G2's own data. Later, the NN learns to map
H&E neighbor compositions (not niche IDs) to biology, enabling true cross-cohort transfer.

**What each command does:**
1. `wsinsight run` — detects cells and extracts H&E features per cell.
2. `wsinsight niche` — builds a kNN graph, learns graph embeddings (DGI), then clusters into niches.
3. `wsinsight import --include niche` — maps Xenium expression onto H&E cells for supervised labeling.

---

### Math 1 · k-Nearest Neighbors (kNN) Graph

For each cell $i$ with feature vector $x_i \in \mathbb{R}^d$, compute pairwise Euclidean distance:
$$
d(x_i, x_j) = \|x_i - x_j\|_2 = \sqrt{\sum_{m=1}^{d}(x_{im} - x_{jm})^2}
$$
Connect cell $i$ to its $k$ nearest cells. The graph adjacency matrix $A \in \{0,1\}^{n \times n}$ is:
$$
A_{ij} = 1 \;\text{if}\; j \in \text{kNN}(i), \quad A_{ij} = 0 \;\text{otherwise}
$$

---

### Math 2 · Deep Graph Infomax (DGI)

DGI learns cell embeddings $h_i \in \mathbb{R}^{d'}$ by maximising mutual information between
local (cell) representations and a global graph summary $s$:
$$
\mathcal{L}_{\text{DGI}}
= \mathbb{E}\!\left[\tfrac{1}{n}\sum_{i} \log D(h_i, s)\right]
+ \mathbb{E}\!\left[\tfrac{1}{n}\sum_{i} \log\!\left(1 - D(\tilde{h}_i, s)\right)\right]
$$
where $D(\cdot,\cdot)$ discriminates real cell–graph pairs from corrupted (shuffled) ones
$\tilde{h}_i$. The learned $h_i$ captures meaningful neighbourhood structure while suppressing noise.

---

### Math 3 · Leiden Clustering (community detection)

Leiden optimises a quality function (modularity $Q$) over the graph partition $\mathcal{P}$:
$$
Q(\mathcal{P}) = \frac{1}{2m}\sum_{c \in \mathcal{P}}
\!\left[e_c - \gamma \frac{K_c^2}{2m}\right]
$$
where $m$ = total edge weight, $e_c$ = internal edge weight of community $c$,
$K_c$ = sum of degrees in $c$, $\gamma$ = resolution parameter.
Higher $Q$ means more edges within communities than expected by chance.

---

### Math 4 · KMeans (centroid clustering)

For $K$ clusters with centroids $\{\mu_k\}$, KMeans minimises total within-cluster variance:
$$
\min_{\{\mu_k\}} \sum_{i=1}^{n} \min_{k} \|x_i - \mu_k\|_2^2
$$
Solved by alternating assignment and centroid update steps.
$x_i$ here is the DGI-learned embedding for cell $i$.

In [ ]:
# Stage 0-D: WSInsight pipeline execution.
#
# Set RUN_WSINSIGHT = True to actually execute the shell commands.
# Default is False so the notebook prints the commands without running them,
# allowing review before committing compute time.
RUN_WSINSIGHT = False

# The run script must handle the two independent runs:
#   1. Group 1 (P1-P6): wsinsight run + niche + import
#   2. Group 2 (P7-P12): wsinsight run + niche + import
# Results land in:
#   outputs/v3/imported-xenium/<stem>.h5ad  for each of the 12 stems.

commands = [
    "# --- Group 1 pipeline (P1-P6, samples S1+S2) ---",
    f"bash {CFG['run_script']} --group g1 --samples {' '.join(G1_STEMS)}",
    "# --- Group 2 pipeline (P7-P12, samples S3+S4) ---",
    f"bash {CFG['run_script']} --group g2 --samples {' '.join(G2_STEMS)}",
]

if RUN_WSINSIGHT:
    import subprocess
    for cmd in commands:
        if cmd.startswith("#"):
            print(cmd)
            continue
        print("Running:", cmd)
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(result.stdout[-2000:] if result.stdout else "(no stdout)")
        if result.returncode != 0:
            print("STDERR:", result.stderr[-1000:])
else:
    print("RUN_WSINSIGHT=False — commands that would be run:")
    for cmd in commands:
        print(" ", cmd)

# Verify expected .h5ad files exist.
available = [CFG["imported_xenium_dir"] / f"{s}.h5ad" for s in ALL_STEMS
             if (CFG["imported_xenium_dir"] / f"{s}.h5ad").exists()]
missing   = [s for s in ALL_STEMS
             if not (CFG["imported_xenium_dir"] / f"{s}.h5ad").exists()]
print(f"\nImported h5ad files found : {len(available)} / {len(ALL_STEMS)}")
if missing:
    print("Missing (run WSInsight first):", missing)

---

## Stage 0-E · Load All h5ad Files

Load all 12 imported h5ad files, attach `sample_id` and `group` metadata, then split into
three working AnnData objects: `adata_g1a`, `adata_g1b`, `adata_g2`.

Each object is preprocessed independently (normalize_total + log1p) to match the
independent niche-clustering runs.

In [ ]:
# Load all 12 h5ad files and split into the three analysis groups.
# Analogy: opening 12 separate spreadsheets and sorting them into three folders.

def load_group(stems, group_tag):
    """Load a list of stems from imported-xenium dir and concatenate into one AnnData."""
    adatas = []
    for stem in stems:
        path = CFG["imported_xenium_dir"] / f"{stem}.h5ad"
        if not path.exists():
            print(f"  WARNING: {path} not found — skipping.")
            continue
        ad = sc.read_h5ad(path)
        ad.obs = ad.obs.copy()
        ad.obs["sample_id"] = stem       # track which sample each cell came from
        ad.obs["group"]     = group_tag  # track which group (g1a / g1b / g2)
        adatas.append(ad)
        print(f"  Loaded {stem}  ({ad.n_obs} cells)")

    if not adatas:
        print(f"  No files found for group {group_tag}. Returning empty AnnData.")
        return sc.AnnData()

    # join='inner' keeps only genes shared across all samples in the group.
    return sc.concat(adatas, join="inner", merge="same",
                     label="cohort_key", keys=stems, index_unique="__")


def preprocess(adata, name):
    """Normalize and log-transform in place. Safe to call once; guarded against double-application."""
    if adata.n_obs == 0:
        print(f"  {name}: empty — skipping preprocessing.")
        return adata
    # Compute QC metrics (total counts, n_genes_by_counts, etc.).
    sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
    # Save raw counts in a dedicated layer for traceability.
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()
    # Guard against double-normalization.
    if "log1p" not in adata.uns:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print(f"  {name}: normalize_total + log1p applied. ({adata.n_obs} cells, {adata.n_vars} genes)")
    else:
        print(f"  {name}: already normalized — skipped.")
    return adata


print("Loading G1A (P1-P3)...")
adata_g1a = preprocess(load_group(G1A_STEMS, "g1a"), "G1A")

print("\nLoading G1B (P4-P6)...")
adata_g1b = preprocess(load_group(G1B_STEMS, "g1b"), "G1B")

print("\nLoading G2 (P7-P12)...")
adata_g2  = preprocess(load_group(G2_STEMS,  "g2"),  "G2")

# Detect niche membership columns (shared across both G1 groups — same wsinsight run).
# G2 has its OWN niche columns from its independent wsinsight run.
def detect_niche_cols(adata, name):
    niche_cols   = sorted([c for c in adata.obs.columns
                           if c.startswith("niche_") and c[6:].isnumeric()],
                          key=lambda x: int(x.split("_")[1]))
    feature_cols = sorted([c for c in adata.obs.columns
                           if c.startswith("niche_feature_normalized_")])
    print(f"  {name}: {len(niche_cols)} niche groups, {len(feature_cols)} feature cols")
    return niche_cols, feature_cols

print("\nDetecting niche columns...")
g1a_niche_cols, g1a_feat_cols = detect_niche_cols(adata_g1a, "G1A")
g1b_niche_cols, g1b_feat_cols = detect_niche_cols(adata_g1b, "G1B")
g2_niche_cols,  g2_feat_cols  = detect_niche_cols(adata_g2,  "G2")

# G1A and G1B must share the same niche column set (same wsinsight run).
if set(g1a_niche_cols) != set(g1b_niche_cols):
    print("WARNING: G1A and G1B niche columns differ — they must come from the same wsinsight run.")
else:
    print("\nG1A and G1B share the same niche column set (expected).")
    # Use G1A's column lists as the canonical G1 column lists.
    g1_niche_cols = g1a_niche_cols
    g1_feat_cols  = g1a_feat_cols

---

## Stage 0-F · Shared GOEA Resources

Download GO ontology and gene-to-GO associations once, then reuse them for G1A, G1B, and G2.

### Math: hypergeometric enrichment

For a GO term with $K$ background genes, given $n$ study genes and $x$ observed overlaps:
$$
p = \sum_{i=x}^{\min(K,n)} \frac{\binom{K}{i}\binom{N-K}{n-i}}{\binom{N}{n}}
$$
Multiple testing corrected by Benjamini–Hochberg FDR (threshold $q < 0.05$).

In [ ]:
# Download (or load cached) GO ontology and gene-to-GO associations.
# These files are shared by G1A, G1B, and G2 GOEA steps.

_goea_cache_dir = CFG["outdir_g1a"]  # store shared files in G1A output dir
obo_path      = download_go_basic_obo(str(_goea_cache_dir / "go-basic.obo"))
gene2go_path  = download_ncbi_associations(str(_goea_cache_dir / "gene2go"))
go_dag        = GODag(obo_path)
objanno       = Gene2GoReader(gene2go_path, taxids=[9606])
ns2assoc      = objanno.get_ns2assc()

print("GOEA resources ready.")
print("  GO obo :", obo_path)
print("  gene2go:", gene2go_path)


# ── Helper: symbol → Entrez mapping ────────────────────────────────────────
def extract_symbol_to_entrez(df):
    """Convert a mygene querymany DataFrame to {SYMBOL: ENTREZ_INT} dict."""
    if isinstance(df, pd.DataFrame):
        x = df.reset_index().rename(columns={"query": "query_symbol"})
    else:
        x = pd.DataFrame(df)
    if "query_symbol" not in x.columns and "query" in x.columns:
        x = x.rename(columns={"query": "query_symbol"})
    x = x.dropna(subset=["entrezgene"]).copy()
    x["query_symbol"] = x["query_symbol"].astype(str).str.upper()
    x["entrezgene"]   = x["entrezgene"].astype(int)
    return dict(zip(x["query_symbol"], x["entrezgene"]))


# ── Helper: run GOEA for one group ─────────────────────────────────────────
def run_goea_for_group(adata, niche_cols, feat_cols, markers_filt, outdir, tag):
    """
    Full GOEA pipeline for one group.

    Parameters
    ----------
    adata         : AnnData with log-normalized expression
    niche_cols    : list of niche membership column names
    feat_cols     : list of H&E feature column names
    markers_filt  : filtered marker gene DataFrame (output of run_de_for_group)
    outdir        : Path — where to write outputs
    tag           : str  — short name used in print statements

    Returns
    -------
    goea_all, goea_sig, goea_summary_df
    """
    # Build GOEA population (gene universe) from this group's variable names.
    universe_symbols = pd.Index(adata.var_names).astype(str).str.upper().unique().tolist()
    uni_q = mg.querymany(universe_symbols, scopes="symbol", fields="entrezgene,symbol",
                         species="human", as_dataframe=True, returnall=False, verbose=False)
    uni_map        = extract_symbol_to_entrez(uni_q)
    population_ids = set(uni_map.values())
    print(f"  {tag}: population Entrez IDs = {len(population_ids)}")

    goea_records, goea_summary = [], []

    for niche_col in niche_cols:
        if markers_filt.empty:
            goea_summary.append({"niche_group": niche_col, "n_sig_terms": 0, "status": "no_markers"})
            continue

        niche_markers = (
            markers_filt.loc[markers_filt["niche_group"] == niche_col, "names"]
            .dropna().astype(str).str.upper().unique().tolist()
        )
        if not niche_markers:
            goea_summary.append({"niche_group": niche_col, "n_sig_terms": 0, "status": "no_markers"})
            continue

        study_q   = mg.querymany(niche_markers, scopes="symbol", fields="entrezgene,symbol",
                                 species="human", as_dataframe=True, returnall=False, verbose=False)
        study_map = extract_symbol_to_entrez(study_q)
        study_ids = {study_map[s] for s in niche_markers
                     if s in study_map and study_map[s] in population_ids}

        if len(study_ids) < CFG["goea_min_genes"]:
            goea_summary.append({"niche_group": niche_col, "n_study_entrez": len(study_ids),
                                  "n_sig_terms": 0, "status": "skipped_low_mapped_ids"})
            continue

        goeaobj = GOEnrichmentStudyNS(population_ids, ns2assoc, go_dag,
                                      propagate_counts=False, alpha=CFG["goea_fdr"],
                                      methods=["fdr_bh"])
        results = goeaobj.run_study(study_ids)

        rows = []
        for r in results:
            rows.append({"niche_group": niche_col, "GO": r.GO, "name": r.name,
                         "namespace": r.NS, "study_count": r.study_count,
                         "study_n": r.study_n, "pop_count": r.pop_count,
                         "pop_n": r.pop_n, "p_uncorrected": r.p_uncorrected,
                         "p_fdr_bh": r.p_fdr_bh, "enrichment": r.enrichment})

        niche_go_df = pd.DataFrame(rows)
        niche_sig   = niche_go_df.loc[niche_go_df["p_fdr_bh"] < CFG["goea_fdr"]]
        goea_records.append(niche_go_df)
        goea_summary.append({"niche_group": niche_col,
                              "n_study_entrez": len(study_ids),
                              "n_sig_terms": int(niche_sig.shape[0]),
                              "status": "ok"})

    goea_all        = pd.concat(goea_records, ignore_index=True) if goea_records else pd.DataFrame()
    goea_sig        = goea_all.loc[goea_all["p_fdr_bh"] < CFG["goea_fdr"]].copy() if not goea_all.empty else pd.DataFrame()
    goea_summary_df = pd.DataFrame(goea_summary)

    outdir.mkdir(parents=True, exist_ok=True)
    goea_all.to_csv(outdir / "goea_all.csv", index=False)
    goea_sig.to_csv(outdir / "goea_sig.csv", index=False)
    goea_summary_df.to_csv(outdir / "goea_summary.csv", index=False)

    print(f"  {tag}: {len(goea_summary_df)} niches processed, "
          f"{int((goea_summary_df.get('status', pd.Series()) == 'ok').sum())} with sig terms.")
    return goea_all, goea_sig, goea_summary_df


# ── Helper: DE for one group ────────────────────────────────────────────────
def run_de_for_group(adata, niche_cols, outdir, tag):
    """
    Wilcoxon DE: in-group (niche membership=1) vs out-group for every niche.

    Returns (markers_all_df, markers_filt_df).
    """
    all_tables, summary = [], []

    for niche_col in tqdm(niche_cols, desc=f"{tag} DE", unit="group"):
        pos_mask = adata.obs[niche_col].fillna(0).astype(int).eq(1)
        n_pos, n_neg = int(pos_mask.sum()), int((~pos_mask).sum())

        if n_pos < CFG["de_min_cells"] or n_neg < CFG["de_min_cells"]:
            summary.append({"niche_group": niche_col, "n_pos": n_pos,
                             "n_neg": n_neg, "status": "skipped_small"})
            continue

        group_col = f"{niche_col}_grp"
        key_added = f"de_{niche_col}"
        adata.obs[group_col] = pd.Categorical(
            np.where(pos_mask, "in", "out"), categories=["in", "out"]
        )
        sc.tl.rank_genes_groups(
            adata, groupby=group_col, groups=["in"], reference="out",
            method="wilcoxon", pts=True, key_added=key_added,
        )
        mk = sc.get.rank_genes_groups_df(adata, group="in", key=key_added)
        mk["niche_group"] = niche_col
        all_tables.append(mk)
        summary.append({"niche_group": niche_col, "n_pos": n_pos,
                         "n_neg": n_neg, "status": "ok"})

    markers_all  = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
    markers_filt = (
        markers_all.query(f"pvals_adj < {CFG['de_pval']} and logfoldchanges > {CFG['de_logfc']}")
        .copy() if not markers_all.empty else pd.DataFrame()
    )
    summary_df = pd.DataFrame(summary)

    outdir.mkdir(parents=True, exist_ok=True)
    markers_all.to_csv(outdir / "markers_all.csv", index=False)
    markers_filt.to_csv(outdir / "markers_filt.csv", index=False)
    summary_df.to_csv(outdir / "de_summary.csv", index=False)

    n_ok = int((summary_df.get("status", pd.Series()) == "ok").sum()) if not summary_df.empty else 0
    print(f"  {tag}: {n_ok}/{len(niche_cols)} niches tested, "
          f"{len(markers_filt)} filtered markers.")
    return markers_all, markers_filt


# ── Helper: LLM niche labeling ──────────────────────────────────────────────
def setup_openai_client():
    """Load OpenAI credentials from .env and return (client, model_name)."""
    from dotenv import find_dotenv, load_dotenv
    from openai import OpenAI

    dotenv_path = find_dotenv(".env", usecwd=True)
    client, model_name = None, "gpt-4.1-mini"

    if dotenv_path:
        load_dotenv(dotenv_path=dotenv_path, override=True)
        api_key    = (os.getenv("OPENAI_API_KEY") or "").strip()
        model_name = (os.getenv("OPENAI_MODEL") or "gpt-4.1-mini").strip()
        base_url   = (os.getenv("OPENAI_BASE_URL") or "").strip() or None
        if base_url and not base_url.startswith(("http://", "https://")):
            base_url = None
        if api_key:
            client = OpenAI(api_key=api_key, base_url=base_url)
    else:
        print("Warning: .env not found. LLM labeling will fall back to placeholder labels.")

    print(f"OpenAI client ready: {client is not None}  model: {model_name}")
    return client, model_name


def ask_label(client, model_name, prompt_text):
    """Request one niche label string from OpenAI."""
    if client is None:
        return "unknown niche"
    response = client.responses.create(
        model=model_name,
        input=(
            "You are a tumor microenvironment expert. "
            "Assign a conventional niche label used in pathology/immunology literature. "
            "The label must be specific enough to distinguish nearby niches in the same dataset. "
            "Avoid generic labels like immune niche or stromal niche unless no specific signal exists. "
            "Prefer biologically grounded wording with a clear modifier. "
            "Return exactly one lowercase label with 3-7 words, no punctuation, no explanation.\n\n"
            + prompt_text
        ),
    )
    out = (getattr(response, "output_text", "") or "").strip()
    if not out:
        return "unknown niche"
    return out.splitlines()[0].strip().strip('"').strip("'").lower()


def run_llm_labeling(goea_sig, niche_cols, outdir, tag, client, model_name):
    """
    For each niche, pick top GO terms and ask the LLM for one human-readable label.
    Returns niche_labels_df with columns [niche_group, niche_label, top_go_terms].
    """
    label_rows, used_labels = [], set()

    # Fallback: if no significant GO terms at all, use placeholder labels.
    if goea_sig.empty:
        niche_labels_df = pd.DataFrame({
            "niche_group": niche_cols,
            "niche_label": [f"label_{g}" for g in niche_cols],
            "top_go_terms": "",
        })
        niche_labels_df.to_csv(outdir / "niche_labels.csv", index=False)
        print(f"  {tag}: no sig GO terms — placeholder labels written.")
        return niche_labels_df

    for niche_col in niche_cols:
        niche_sig = goea_sig.loc[goea_sig["niche_group"] == niche_col].copy()
        if niche_sig.empty:
            continue

        top_terms = niche_sig.nsmallest(CFG["llm_top_terms"], "p_fdr_bh")
        top_terms["term_text"] = (top_terms["GO"] + " | " +
                                   top_terms["name"] + " (" + top_terms["namespace"] + ")")
        top_go_str = "; ".join(top_terms["term_text"].tolist())

        # Build qualifier tokens for disambiguation fallback.
        qualifier_tokens = []
        for term_name in top_terms["name"].astype(str).tolist():
            for w in re.findall(r"[a-z]+", term_name.lower()):
                if w not in {"process", "regulation", "positive", "negative",
                              "cell", "cells", "response", "pathway"} and len(w) >= 4:
                    qualifier_tokens.append(w)
        fallback_q = qualifier_tokens[0] if qualifier_tokens else niche_col.replace("_", "")

        prompt_text = (
            f"Niche group ID: {niche_col}. "
            f"Top enriched GO terms: {top_go_str}. "
            f"Labels already assigned: {sorted(used_labels) if used_labels else 'none yet'}. "
            "Return one conventional tumor-microenvironment niche label."
        )

        label = re.sub(r"[^a-z0-9 ]+", " ",
                       ask_label(client, model_name, prompt_text).lower()).strip()
        label = " ".join(label.split())

        # One retry if duplicate.
        if label in used_labels:
            retry_prompt = prompt_text + f" '{label}' is already used. Choose a different label."
            label = re.sub(r"[^a-z0-9 ]+", " ",
                           ask_label(client, model_name, retry_prompt).lower()).strip()
            label = " ".join(label.split())
        if label in used_labels:
            label = f"{label} {fallback_q}".strip()

        used_labels.add(label)
        label_rows.append({"niche_group": niche_col, "niche_label": label,
                            "top_go_terms": top_go_str})

    niche_labels_df = pd.DataFrame(label_rows).sort_values("niche_group").reset_index(drop=True)
    niche_labels_df.to_csv(outdir / "niche_labels.csv", index=False)
    print(f"  {tag}: labeled {len(niche_labels_df)} niche groups.")
    print(niche_labels_df[["niche_group", "niche_label"]].to_string(index=False))
    return niche_labels_df


print("\nAll helper functions defined and GOEA resources ready.")

---

## Stage 1 · G1A — Build Niche Labels (Marker Genes → GOEA → LLM)

G1A = P1–P3 (sample S1, all three tissue levels).
This is the **label-construction** split. Xenium expression from G1A is used to derive
human-readable niche type names. These labels become the training targets for the NN in Stage 3.

**Why G1A only?**  
G1B is reserved as an independent validation for label consistency (Stage 2).
Using G1A and G1B together to derive labels would conflate the construction and validation sets.

### Filter criteria for marker genes

$$
p_{\text{adj}}(g) < 0.05 \quad \text{and} \quad \log_2 FC(g) > 0.25
$$

In [ ]:
# Stage 1-A: Differential expression on G1A.
# For each niche group: in-group cells (membership=1) vs all other cells.
print("Stage 1 — DE on G1A...")
g1a_markers_all, g1a_markers_filt = run_de_for_group(
    adata_g1a, g1_niche_cols, CFG["outdir_g1a"], "G1A"
)
print(f"  Total marker rows : {len(g1a_markers_all)}")
print(f"  Filtered marker rows: {len(g1a_markers_filt)}")

### G1A — GOEA

Run GO enrichment on G1A marker genes to identify dominant biological processes per niche.

In [ ]:
# Stage 1-B: GOEA on G1A marker genes.
print("Stage 1 — GOEA on G1A...")
g1a_goea_all, g1a_goea_sig, g1a_goea_summary = run_goea_for_group(
    adata_g1a, g1_niche_cols, g1_feat_cols,
    g1a_markers_filt, CFG["outdir_g1a"], "G1A"
)
print("\nG1A GOEA summary:")
print(g1a_goea_summary.to_string(index=False))

### G1A — LLM Niche Labeling

Send top GO terms per niche to OpenAI and receive one conventional niche type name per group
(e.g., *tumor proliferative*, *immune interferon response*, *TLS lymphoid aggregate*).

In [ ]:
# Stage 1-C: LLM niche labeling from G1A GOEA results.
# Credentials are loaded from a .env file; if absent, placeholder labels are used.
print("Stage 1 — LLM labeling on G1A...")
llm_client, llm_model = setup_openai_client()

g1a_niche_labels = run_llm_labeling(
    g1a_goea_sig, g1_niche_cols,
    CFG["outdir_g1a"], "G1A",
    llm_client, llm_model,
)

# The G1A label table is the canonical label definition for all of G1.
# It maps: niche_group (e.g. niche_2)  →  niche_label (e.g. "TLS lymphoid")
print("\nG1A canonical niche labels (used for training in Stage 3):")
print(g1a_niche_labels[["niche_group", "niche_label"]].to_string(index=False))

---

## Stage 2 · G1B — Independent Label Consistency Check

G1B = P4–P6 (sample S2). G1B shares the **same niche ID space** as G1A (both were processed
in the same WSInsight run). However, G1B's Xenium data is completely independent from G1A.

We independently run DE → GOEA → LLM on G1B, producing its own label table.

**Validation question:**  
> Does niche_k receive the same human-readable label in G1B as in G1A?

We measure agreement with a **confusion matrix** (niche-level, not cell-level) and a
**label-agreement rate** across the shared niche ID set.

If agreement is high, it confirms that cell-neighbor composition (H&E features) is a
stable and biologically meaningful representation across different patient samples.

In [ ]:
# Stage 2-A: DE on G1B (independent of G1A).
print("Stage 2 — DE on G1B...")
g1b_markers_all, g1b_markers_filt = run_de_for_group(
    adata_g1b, g1_niche_cols, CFG["outdir_g1b"], "G1B"
)
print(f"  Total marker rows   : {len(g1b_markers_all)}")
print(f"  Filtered marker rows: {len(g1b_markers_filt)}")

In [ ]:
# Stage 2-B: GOEA on G1B.
print("Stage 2 — GOEA on G1B...")
g1b_goea_all, g1b_goea_sig, g1b_goea_summary = run_goea_for_group(
    adata_g1b, g1_niche_cols, g1_feat_cols,
    g1b_markers_filt, CFG["outdir_g1b"], "G1B"
)
print("\nG1B GOEA summary:")
print(g1b_goea_summary.to_string(index=False))

In [ ]:
# Stage 2-C: LLM labeling on G1B (fully independent from G1A).
print("Stage 2 — LLM labeling on G1B...")
g1b_niche_labels = run_llm_labeling(
    g1b_goea_sig, g1_niche_cols,
    CFG["outdir_g1b"], "G1B",
    llm_client, llm_model,
)

### Stage 2-D · Label Consistency Evaluation (AUC / Confusion Matrix)

For each niche ID present in both G1A and G1B, compare whether both GOEA+LLM pipelines
assigned the same human-readable label.

We treat G1A labels as the reference ("ground truth") and G1B labels as the "predictions".
The confusion matrix is at the **niche level** (one row per shared niche ID).

In [ ]:
# Stage 2-D: Label consistency evaluation between G1A and G1B.
# This confirms that the same niche ID receives the same biological interpretation
# when derived from two completely independent Xenium experiments.

# Merge on niche_group (shared niche IDs).
consistency_df = g1a_niche_labels[["niche_group", "niche_label"]].rename(
    columns={"niche_label": "label_g1a"}
).merge(
    g1b_niche_labels[["niche_group", "niche_label"]].rename(
        columns={"niche_label": "label_g1b"}
    ),
    on="niche_group", how="inner",
)

# Mark exact label agreement.
consistency_df["exact_match"] = consistency_df["label_g1a"] == consistency_df["label_g1b"]

n_shared = len(consistency_df)
n_match  = int(consistency_df["exact_match"].sum())
agree_rate = n_match / max(n_shared, 1)

print(f"Shared niche IDs : {n_shared}")
print(f"Exact matches    : {n_match}")
print(f"Agreement rate   : {agree_rate:.2%}")
print()
print(consistency_df.to_string(index=False))

# Save consistency table.
consistency_df.to_csv(CFG["nn_dir"] / "g1ab_label_consistency.csv", index=False)

# Confusion matrix at niche level (rows = G1A label, columns = G1B label).
if n_shared >= 2:
    all_labels_for_cm = sorted(set(consistency_df["label_g1a"]) | set(consistency_df["label_g1b"]))
    cm = confusion_matrix(consistency_df["label_g1a"], consistency_df["label_g1b"],
                          labels=all_labels_for_cm)
    cm_df = pd.DataFrame(cm, index=all_labels_for_cm, columns=all_labels_for_cm)

    fig, ax = plt.subplots(figsize=(max(6, len(all_labels_for_cm)), max(5, len(all_labels_for_cm))))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=ax,
                cbar_kws={"label": "Niche count"})
    ax.set_xlabel("G1B label (independent)", fontsize=11)
    ax.set_ylabel("G1A label (reference)", fontsize=11)
    ax.set_title("G1A vs G1B niche label consistency\n"
                 "(same niche IDs, independent Xenium derivation)", fontsize=12)
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)
    fig.tight_layout()
    fig.savefig(CFG["nn_dir"] / "g1ab_label_consistency_cm.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", CFG["nn_dir"] / "g1ab_label_consistency_cm.png")
else:
    print("Too few shared niches to plot confusion matrix.")

---

## Stage 3 · Train Neural Network on G1 (H&E Features → Niche Labels)

### Scientific rationale

Stage 2 confirmed that cell-neighbor composition (H&E features) reliably identifies biological
niche type across G1A and G1B. Now we train a neural network that learns:

$$
f_\theta(\phi_i) \rightarrow y_i
$$

where $\phi_i \in \mathbb{R}^{2d}$ is the per-niche mean+std feature vector and $y_i$ is the
G1A-derived human-readable label.

Training uses **all of G1** (P1–P6) with the G1A-derived labels (which apply to both G1A and G1B
since they share the same niche ID space).

### NN architecture

A simple MLP:
$$
h^{(1)} = \text{ReLU}(W_1 z + b_1), \quad
h^{(2)} = \text{ReLU}(W_2 h^{(1)} + b_2), \quad
\hat{y} = \text{softmax}(W_3 h^{(2)} + b_3)
$$

Cross-entropy loss:
$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log \hat{p}(y_i | \phi_i)
$$

In [ ]:
# Stage 3-A: Build per-sample, per-niche feature vectors for all of G1.
# Each vector = [mean_1, ..., mean_d, std_1, ..., std_d] over cells in that niche.
# Label comes from G1A-derived labels (same niche IDs apply to G1B).

# Merge G1A + G1B into one AnnData with sample_id tracking.
adata_g1 = sc.concat(
    [adata_g1a, adata_g1b],
    join="inner", merge="same",
    label="cohort_key",
    keys=["g1a", "g1b"],
    index_unique="__"
)
# Restore sample_id (sc.concat may overwrite it with cohort_key)
adata_g1.obs["sample_id"] = (
    adata_g1a.obs["sample_id"].tolist() +
    adata_g1b.obs["sample_id"].tolist()
)

# Label map: niche_group → human-readable label (from G1A derivation).
label_map_g1 = dict(zip(g1a_niche_labels["niche_group"], g1a_niche_labels["niche_label"]))

obs_g1 = adata_g1.obs.copy()
vector_rows = []

for sample_id in G1_STEMS:
    sample_obs = obs_g1.loc[obs_g1["sample_id"] == sample_id].copy()
    if sample_obs.empty:
        print(f"  WARNING: no cells found for {sample_id} — skipping.")
        continue

    # Identify which sub-group this sample belongs to (g1a or g1b).
    sub_group = "g1a" if sample_id in G1A_STEMS else "g1b"

    for niche_col in g1_niche_cols:
        if niche_col not in label_map_g1:
            continue  # niche not labeled (rare)

        # Select cells assigned to this niche in this sample.
        membership = pd.to_numeric(sample_obs[niche_col], errors="coerce").fillna(0).astype(int)
        mask  = membership.eq(1)
        n_cells = int(mask.sum())
        if n_cells == 0:
            continue

        # Compute mean and std of all H&E feature columns for this niche.
        feat = sample_obs.loc[mask, g1_feat_cols].apply(pd.to_numeric, errors="coerce")
        mean_vec = feat.mean(axis=0, skipna=True).to_numpy(dtype=float)
        std_vec  = feat.std(axis=0, ddof=1, skipna=True).to_numpy(dtype=float)
        vec = np.concatenate([mean_vec, std_vec]).tolist()

        vector_rows.append({
            "sample_id":   sample_id,
            "sub_group":   sub_group,
            "niche_group": niche_col,
            "niche_label": label_map_g1[niche_col],
            "n_cells":     n_cells,
            "vector":      vec,
        })

g1_vectors_df = pd.DataFrame(vector_rows)
g1_vectors_df.to_csv(CFG["nn_dir"] / "g1_vectors.csv", index=False)

print(f"G1 vector rows built: {len(g1_vectors_df)}")
print(f"Feature dimension  : {len(g1_feat_cols)} features → vector length {2*len(g1_feat_cols)}")
print(f"Niche label classes: {sorted(g1_vectors_df['niche_label'].unique())}")

In [ ]:
# Stage 3-B: Encode labels, scale features, build PyTorch datasets.
# Standardization formula: z = (x - μ) / σ, fit on G1 training data only.

# We use a 80/20 split within G1 for internal train/val monitoring.
# The real held-out test is G2 (Stage 4).
from sklearn.model_selection import train_test_split

# Encode niche labels as integer class IDs.
g1_labels = sorted(g1_vectors_df["niche_label"].unique().tolist())
label_encoder = LabelEncoder().fit(g1_labels)
g1_vectors_df["class_id"] = label_encoder.transform(g1_vectors_df["niche_label"])

# Convert list-of-lists to a float32 matrix.
X_g1 = np.vstack(g1_vectors_df["vector"].tolist()).astype(np.float32)
X_g1 = np.nan_to_num(X_g1, nan=0.0, posinf=0.0, neginf=0.0)
y_g1 = g1_vectors_df["class_id"].to_numpy(dtype=np.int64)

# 80 / 20 train-val split (stratified to keep class balance).
idx_all = np.arange(len(g1_vectors_df))
idx_tr, idx_val = train_test_split(
    idx_all, test_size=0.2, random_state=CFG["seed"],
    stratify=y_g1 if len(np.unique(y_g1)) > 1 else None
)

# Fit scaler on training rows only to prevent data leakage.
scaler = StandardScaler().fit(X_g1[idx_tr])
X_g1_s = scaler.transform(X_g1).astype(np.float32)

X_tr, y_tr = X_g1_s[idx_tr], y_g1[idx_tr]
X_val, y_val = X_g1_s[idx_val], y_g1[idx_val]

class_names = list(label_encoder.classes_)
num_classes = len(class_names)
input_dim   = X_tr.shape[1]

if num_classes < 2:
    raise RuntimeError("Need at least 2 label classes to train a classifier.")

# Build PyTorch datasets.
train_dataset = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

print("Model inputs ready.")
print(f"  Input dimension : {input_dim}")
print(f"  Num classes     : {num_classes}  →  {class_names}")
print(f"  Training rows   : {len(X_tr)}")
print(f"  Validation rows : {len(X_val)}")

In [ ]:
# Stage 3-C: Define MLP and train on G1  (or reload from disk).
#
# ── USER FLAG ──────────────────────────────────────────────────────────────
# RELOAD_MODEL = False  →  train from scratch using Stage 3-B outputs.
# RELOAD_MODEL = True   →  skip training; load saved model + scaler + encoder.
#                          Use this when resuming a new session without retraining.
# ───────────────────────────────────────────────────────────────────────────
RELOAD_MODEL = False   # <── change to True to reload instead of retraining

import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("RELOAD_MODEL :", RELOAD_MODEL)


class NicheMLP(nn.Module):
    """
    Simple feed-forward MLP for niche label classification.
    Architecture: input → [hidden layers with BatchNorm + ReLU + Dropout] → output
    """

    def __init__(self, input_dim, hidden_sizes, num_classes, dropout):
        super().__init__()
        layers = []
        prev_size = input_dim
        for h in hidden_sizes:
            layers += [
                nn.Linear(prev_size, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            prev_size = h
        layers.append(nn.Linear(prev_size, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


if RELOAD_MODEL:
    # ── Load saved artifacts ────────────────────────────────────────────────
    model_path  = CFG["nn_dir"] / "niche_mlp.pt"
    scaler_path = CFG["nn_dir"] / "scaler.pkl"
    le_path     = CFG["nn_dir"] / "label_encoder.pkl"

    missing = [p for p in [model_path, scaler_path, le_path] if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"Cannot reload — missing files: {missing}. "
            "Run with RELOAD_MODEL=False first to train and save."
        )

    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    with open(le_path, "rb") as f:
        label_encoder = pickle.load(f)

    class_names = list(label_encoder.classes_)
    num_classes  = len(class_names)
    # Infer input_dim from the first Linear layer weight in the saved state dict.
    state     = torch.load(model_path, map_location=device)
    input_dim = state["net.0.weight"].shape[1]

    model = NicheMLP(
        input_dim    = input_dim,
        hidden_sizes = CFG["nn_hidden"],
        num_classes  = num_classes,
        dropout      = CFG["nn_dropout"],
    ).to(device)
    model.load_state_dict(state)
    model.eval()

    print("Model reloaded from disk.")
    print(f"  Classes   : {class_names}")
    print(f"  Input dim : {input_dim}")

else:
    # ── Train from scratch ──────────────────────────────────────────────────

    # Instantiate model using dimensions from Stage 3-B.
    model = NicheMLP(
        input_dim    = input_dim,
        hidden_sizes = CFG["nn_hidden"],
        num_classes  = num_classes,
        dropout      = CFG["nn_dropout"],
    ).to(device)

    optimizer    = torch.optim.Adam(model.parameters(),
                                    lr=CFG["nn_lr"],
                                    weight_decay=CFG["nn_weight_decay"])
    criterion    = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_dataset,
                              batch_size=CFG["nn_batch_size"],
                              shuffle=True, drop_last=False)

    # Training loop: one full pass over training data per epoch.
    history = {"epoch": [], "train_loss": [], "val_acc": []}

    for epoch in range(1, CFG["nn_epochs"] + 1):
        # Training pass.
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_loss = epoch_loss / len(train_dataset)

        # Validation pass (no gradient computation needed).
        model.eval()
        with torch.no_grad():
            val_logits = model(torch.tensor(X_val, dtype=torch.float32, device=device))
            val_pred   = val_logits.argmax(dim=1).cpu().numpy()
        val_acc = accuracy_score(y_val, val_pred)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{CFG['nn_epochs']}  "
                  f"loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    # Save all artifacts to disk so Stage 4 (or a future session) can reload them.
    torch.save(model.state_dict(), CFG["nn_dir"] / "niche_mlp.pt")
    with open(CFG["nn_dir"] / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    with open(CFG["nn_dir"] / "label_encoder.pkl", "wb") as f:
        pickle.dump(label_encoder, f)

    print("\nTraining complete.")
    print(f"  Model saved : {CFG['nn_dir'] / 'niche_mlp.pt'}")
    print(f"  Final val acc: {history['val_acc'][-1]:.4f}")

    # Plot training curve.
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(CFG["nn_dir"] / "training_history.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(hist_df["epoch"], hist_df["train_loss"])
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss")
    axes[0].set_title("G1 Training Loss")
    axes[1].plot(hist_df["epoch"], hist_df["val_acc"], color="orange")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_title("G1 Internal Validation Accuracy")
    fig.tight_layout()
    fig.savefig(CFG["nn_dir"] / "g1_training_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

---

## Stage 4 · Cross-Group Transfer: Predict G2 Niche Types from H&E Features

### Key design

G2 was processed by an **independent** WSInsight run, so its niche IDs have no direct correspondence
to G1 niche IDs. The NN trained on G1 does **not** use niche IDs — it uses the continuous H&E
neighbor-composition feature vector $\phi_i$, which encodes biology regardless of niche numbering.

**Stage 4 workflow:**
1. Build G2 feature vectors (same formula as G1).
2. Apply the G1-trained NN → predicted G1-space niche labels for G2 niches.
3. Independently run DE → GOEA → LLM on G2 → ground-truth G2 niche labels.
4. Compare predictions vs ground truth → AUC / confusion matrix.

If the NN generalizes, it means H&E neighbor composition captures transferable biology.

In [ ]:
# Stage 4-A: Build G2 feature vectors using the same mean+std formula.
# G2 uses g2_feat_cols (feature columns from G2's own wsinsight run).
# We must check that G2 features are in the same semantic space as G1 features.
# If the wsinsight run uses consistent feature extraction, the column names will match.

obs_g2 = adata_g2.obs.copy()
g2_vector_rows = []

# Use G1 feature column names as the reference; only keep G2 columns that match.
shared_feat_cols = [c for c in g1_feat_cols if c in g2_feat_cols]
print(f"Shared feature columns between G1 and G2: {len(shared_feat_cols)} / "
      f"(G1={len(g1_feat_cols)}, G2={len(g2_feat_cols)})")
if len(shared_feat_cols) < len(g1_feat_cols):
    print("WARNING: G2 is missing some G1 feature columns. "
          "This may reduce prediction quality.")

for sample_id in G2_STEMS:
    sample_obs = obs_g2.loc[obs_g2["sample_id"] == sample_id].copy()
    if sample_obs.empty:
        print(f"  WARNING: {sample_id} not found in G2 AnnData — skipping.")
        continue

    for niche_col in g2_niche_cols:
        membership = pd.to_numeric(sample_obs[niche_col], errors="coerce").fillna(0).astype(int)
        mask    = membership.eq(1)
        n_cells = int(mask.sum())
        if n_cells == 0:
            continue

        feat = sample_obs.loc[mask, shared_feat_cols].apply(pd.to_numeric, errors="coerce")
        mean_vec = feat.mean(axis=0, skipna=True).to_numpy(dtype=float)
        std_vec  = feat.std(axis=0, ddof=1, skipna=True).to_numpy(dtype=float)

        # Pad zeros for any G1 feature columns absent in G2.
        if len(shared_feat_cols) < len(g1_feat_cols):
            full_mean = np.zeros(len(g1_feat_cols), dtype=float)
            full_std  = np.zeros(len(g1_feat_cols), dtype=float)
            for i, col in enumerate(g1_feat_cols):
                if col in shared_feat_cols:
                    j = shared_feat_cols.index(col)
                    full_mean[i] = mean_vec[j]
                    full_std[i]  = std_vec[j]
            vec = np.concatenate([full_mean, full_std]).tolist()
        else:
            vec = np.concatenate([mean_vec, std_vec]).tolist()

        g2_vector_rows.append({
            "sample_id":   sample_id,
            "niche_group": niche_col,  # G2's own niche IDs
            "n_cells":     n_cells,
            "vector":      vec,
        })

g2_vectors_df = pd.DataFrame(g2_vector_rows)
g2_vectors_df.to_csv(CFG["nn_dir"] / "g2_vectors.csv", index=False)
print(f"\nG2 vector rows built: {len(g2_vectors_df)}")

In [ ]:
# Stage 4-B: Run NN on G2 to predict G1-space niche labels.
# The scaler and label_encoder are the G1-trained objects from Stage 3.

model.eval()  # switch to inference mode (disables dropout / batchnorm training behavior)

X_g2 = np.vstack(g2_vectors_df["vector"].tolist()).astype(np.float32)
X_g2 = np.nan_to_num(X_g2, nan=0.0, posinf=0.0, neginf=0.0)

# Apply the G1 scaler (fit on G1 training data — never refit on G2).
X_g2_s = scaler.transform(X_g2).astype(np.float32)

with torch.no_grad():
    logits_g2 = model(torch.tensor(X_g2_s, dtype=torch.float32, device=device))
    probs_g2  = torch.softmax(logits_g2, dim=1).cpu().numpy()
    preds_g2  = probs_g2.argmax(axis=1)

# Decode integer predictions to human-readable labels.
pred_labels_g2 = label_encoder.inverse_transform(preds_g2)

# Attach predictions to the G2 vector table.
g2_vectors_df["predicted_label"] = pred_labels_g2
for idx, cls in enumerate(class_names):
    g2_vectors_df[f"prob_{cls}"] = probs_g2[:, idx]

g2_vectors_df.to_csv(CFG["nn_dir"] / "g2_predictions.csv", index=False)

print("G2 NN predictions complete.")
print("Predicted label distribution:")
print(g2_vectors_df["predicted_label"].value_counts().to_string())

### Stage 4-C · Derive G2 Ground-Truth Labels (Independent GOEA+LLM)

G2's Xenium data is now used to independently derive human-readable niche labels for G2.
These labels serve as the **held-out ground truth** for evaluating the NN predictions.

This is the cleanest possible validation: the ground truth was derived from a completely
different cohort (different patients, different wsinsight run, independent GOEA+LLM).

In [ ]:
# Stage 4-C-i: DE on G2.
print("Stage 4 — DE on G2...")
g2_markers_all, g2_markers_filt = run_de_for_group(
    adata_g2, g2_niche_cols, CFG["outdir_g2"], "G2"
)
print(f"  Total marker rows   : {len(g2_markers_all)}")
print(f"  Filtered marker rows: {len(g2_markers_filt)}")

In [ ]:
# Stage 4-C-ii: GOEA on G2.
print("Stage 4 — GOEA on G2...")
g2_goea_all, g2_goea_sig, g2_goea_summary = run_goea_for_group(
    adata_g2, g2_niche_cols, g2_feat_cols,
    g2_markers_filt, CFG["outdir_g2"], "G2"
)
print("\nG2 GOEA summary:")
print(g2_goea_summary.to_string(index=False))

In [ ]:
# Stage 4-C-iii: LLM labeling on G2 (fully independent — different niche IDs from G1).
print("Stage 4 — LLM labeling on G2...")
g2_niche_labels = run_llm_labeling(
    g2_goea_sig, g2_niche_cols,
    CFG["outdir_g2"], "G2",
    llm_client, llm_model,
)

# Attach G2 ground-truth labels to the prediction table.
g2_label_map = dict(zip(g2_niche_labels["niche_group"], g2_niche_labels["niche_label"]))
g2_vectors_df["true_label_g2"] = g2_vectors_df["niche_group"].map(g2_label_map)
g2_vectors_df.to_csv(CFG["nn_dir"] / "g2_predictions_with_truth.csv", index=False)

print("\nG2 true label distribution:")
print(g2_vectors_df["true_label_g2"].value_counts().to_string())

### Stage 4-D · Cross-Group Evaluation: Confusion Matrix + AUC

Compare the NN's predicted labels against the G2-derived ground truth.

Because G2 labels come from a different label space (different LLM calls), we evaluate
**only on the niche label types that appear in both G1 and G2 ground truths** (seen labels).
Rows with G2-only label types are reported separately as coverage.

---

### Math 5 · Confusion Matrix

For a $C$-class problem, the confusion matrix $M \in \mathbb{Z}^{C \times C}$ is:
$$
M_{rc} = \text{number of samples with true class } r \text{ predicted as class } c
$$
Diagonal $M_{cc}$ = correct predictions. Row-normalised version:
$$
\hat{M}_{rc} = \frac{M_{rc}}{\sum_{c'} M_{rc'}}
$$
Per-class derived metrics (treating class $k$ as positive, rest as negative):
$$
\text{Precision}_k = \frac{M_{kk}}{\sum_r M_{rk}}, \quad
\text{Recall}_k    = \frac{M_{kk}}{\sum_c M_{kc}}, \quad
F_1^{(k)}           = \frac{2 \cdot \text{Precision}_k \cdot \text{Recall}_k}
                           {\text{Precision}_k + \text{Recall}_k}
$$
Macro-$F_1$ averages $F_1^{(k)}$ equally across all $C$ classes:
$$
F_1^{\text{macro}} = \frac{1}{C}\sum_{k=1}^{C} F_1^{(k)}
$$

---

### Math 6 · ROC Curve and AUC

For class $k$ (one-vs-rest), at decision threshold $t$ on predicted probability $\hat{p}_k$:
$$
\text{TPR}_k(t) = \frac{\text{TP}_k(t)}{\text{TP}_k(t)+\text{FN}_k(t)},
\qquad
\text{FPR}_k(t) = \frac{\text{FP}_k(t)}{\text{FP}_k(t)+\text{TN}_k(t)}
$$
The ROC curve traces $\bigl(\text{FPR}_k(t),\, \text{TPR}_k(t)\bigr)$ as $t$ decreases 1 → 0.
AUC is the area under this curve (range $[0,1]$; random classifier $\approx 0.5$):
$$
\text{AUC}_k = \int_0^1 \text{TPR}_k\!\left(\text{FPR}_k^{-1}(u)\right)du
$$
**Micro-average** pools all class binary decisions before computing TPR/FPR.  
**Macro-average** averages the per-class AUC scores equally across all $C$ classes.

In [ ]:
# Stage 4-D: Cross-group evaluation.
# G2 always has ≥2 niche types (multiple niche groups × multiple GOEA+LLM labels),
# so all metric computation runs unconditionally — no conditional guard needed.

# Drop rows with no G2 ground truth (niche had no GOEA sig terms → no LLM label).
eval_df = g2_vectors_df.dropna(subset=["true_label_g2"]).copy()

# Separate rows whose true label was seen in G1 training vs. unseen (new biology in G2).
seen_mask   = eval_df["true_label_g2"].isin(class_names)
eval_seen   = eval_df.loc[seen_mask].copy()
eval_unseen = eval_df.loc[~seen_mask].copy()

print(f"G2 niche rows with ground truth    : {len(eval_df)}")
print(f"  Rows with seen labels (evaluable) : {len(eval_seen)}")
print(f"  Rows with unseen labels (excluded): {len(eval_unseen)}")

# Encode ground-truth labels using the G1 label encoder.
y_true_g2 = label_encoder.transform(eval_seen["true_label_g2"])
y_pred_g2 = label_encoder.transform(eval_seen["predicted_label"])
prob_cols  = [f"prob_{c}" for c in class_names]
prob_mat   = eval_seen[prob_cols].to_numpy(dtype=np.float32)

# ── Scalar metrics ─────────────────────────────────────────────────────────
acc_g2 = accuracy_score(y_true_g2, y_pred_g2)
f1_g2  = f1_score(y_true_g2, y_pred_g2, average="macro", zero_division=0)
print(f"\nCross-group accuracy  (G2 eval): {acc_g2:.4f}")
print(f"Cross-group macro-F1  (G2 eval): {f1_g2:.4f}")

# ── Classification report (per-class precision / recall / F1) ──────────────
# Matches the reference notebook convention; also written to disk.
cm_labels   = sorted(set(y_true_g2) | set(y_pred_g2))
cm_names    = label_encoder.inverse_transform(cm_labels)
report_dict = classification_report(
    y_true_g2, y_pred_g2,
    labels=cm_labels, target_names=cm_names,
    output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
report_df.to_csv(CFG["nn_dir"] / "g2_classification_report.csv")
print("\nPer-class classification report (G2):")
print(report_df.to_string())

# ── Confusion matrix ────────────────────────────────────────────────────────
cm_g2      = confusion_matrix(y_true_g2, y_pred_g2, labels=cm_labels)
cm_g2_norm = confusion_matrix(y_true_g2, y_pred_g2, labels=cm_labels, normalize="true")
cm_g2_df   = pd.DataFrame(cm_g2_norm * 100, index=cm_names, columns=cm_names)

cm_g2_df.to_csv(CFG["nn_dir"] / "g2_confusion_matrix.csv")
pd.DataFrame(cm_g2, index=cm_names, columns=cm_names).to_csv(
    CFG["nn_dir"] / "g2_confusion_matrix_counts.csv"
)

fig_cm, ax_cm = plt.subplots(figsize=(max(6, len(cm_names)), max(5, len(cm_names))))
sns.heatmap(cm_g2_df, annot=True, fmt=".1f", cmap="Blues", ax=ax_cm,
            vmin=0, vmax=100, cbar_kws={"label": "Row-normalized %", "shrink": 0.78})
ax_cm.set_xlabel("Predicted label (G1-trained NN)", fontsize=11)
ax_cm.set_ylabel("True label (G2 GOEA+LLM)", fontsize=11)
ax_cm.set_title("Cross-group evaluation: G2 predicted vs G2 ground truth\n"
                "(row-normalized %, seen-label subset)", fontsize=12)
ax_cm.tick_params(axis="x", rotation=45, labelsize=9)
ax_cm.tick_params(axis="y", rotation=0,  labelsize=9)
fig_cm.tight_layout()
fig_cm.savefig(CFG["nn_dir"] / "g2_confusion_matrix_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", CFG["nn_dir"] / "g2_confusion_matrix_heatmap.png")

# ── ROC-AUC (one-vs-rest per class + micro-average) ────────────────────────
y_bin_g2    = label_binarize(y_true_g2, classes=np.arange(num_classes))
roc_records = []
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

for idx in np.unique(y_true_g2):
    if y_bin_g2[:, idx].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_bin_g2[:, idx], prob_mat[:, idx])
    roc_auc_val = auc(fpr, tpr)
    roc_records.append({"class": class_names[idx], "roc_auc": roc_auc_val})
    ax_roc.plot(fpr, tpr, alpha=0.75, label=f"{class_names[idx]} (AUC={roc_auc_val:.3f})")

# Micro-average: pool all binary decisions, then compute one TPR/FPR curve.
fpr_micro, tpr_micro, _ = roc_curve(y_bin_g2.ravel(), prob_mat.ravel())
auc_micro = auc(fpr_micro, tpr_micro)
ax_roc.plot(fpr_micro, tpr_micro, "k-", lw=2.5, label=f"micro-avg (AUC={auc_micro:.3f})")
roc_records.append({"class": "overall_micro", "roc_auc": auc_micro})

ax_roc.plot([0, 1], [0, 1], "--", color="gray", lw=1)
ax_roc.set_xlabel("False Positive Rate", fontsize=11)
ax_roc.set_ylabel("True Positive Rate", fontsize=11)
ax_roc.set_title("Cross-group ROC curves (G2 one-vs-rest)", fontsize=12)
ax_roc.legend(loc="lower right", fontsize=9)
fig_roc.tight_layout()
fig_roc.savefig(CFG["nn_dir"] / "g2_roc_curves.png", dpi=300, bbox_inches="tight")
plt.show()

roc_df = pd.DataFrame(roc_records)
roc_df.to_csv(CFG["nn_dir"] / "g2_roc_auc.csv", index=False)
print("\nROC-AUC per class (G2 cross-group):")
print(roc_df.to_string(index=False))

# ── Coverage summary ────────────────────────────────────────────────────────
# All metric variables are always defined above (no conditional guard).
coverage = pd.DataFrame([{
    "g2_niche_rows_total":    len(g2_vectors_df),
    "g2_rows_with_truth":     len(eval_df),
    "g2_rows_seen_labels":    len(eval_seen),
    "g2_rows_unseen_labels":  len(eval_unseen),
    "accuracy":               acc_g2,
    "macro_f1":               f1_g2,
    "roc_auc_micro":          auc_micro,
}])
coverage.to_csv(CFG["nn_dir"] / "g2_eval_coverage.csv", index=False)
print("\nEvaluation coverage summary:")
print(coverage.to_string(index=False))

---

## Discussion and Conclusion

### What was validated

| Claim | Evidence |
|-------|----------|
| Niche IDs encode biology consistently within a cohort | Stage 2: G1A vs G1B label agreement (AUC/CM) |
| H&E neighbor composition transfers biology across cohorts | Stage 4: G1-trained NN on G2, validated by independent G2 GOEA+LLM |
| Xenium is not required at inference time | Stage 4: predictions use only H&E features |

### Interpretation

- **High G1A/G1B agreement** in Stage 2 means the biological niche structure is consistent
  within the same wsinsight-defined niche space. The neighbor composition is a reliable proxy.

- **High G2 accuracy** in Stage 4 means the learned mapping from H&E composition to niche type
  generalizes across patient cohorts processed independently.

- **Unseen labels** (G2 niche types absent from G1) indicate biology not represented in the G1
  training set. Enlarging G1 or adding more diverse samples would extend coverage.

### Outputs

```
outputs/v3/
  v2_g1a_goea/    markers_all.csv, markers_filt.csv, goea_all.csv, goea_sig.csv, niche_labels.csv
  v2_g1b_goea/    (same structure — independent G1B derivation)
  v2_g2_goea/     (same structure — independent G2 derivation)
  v2_nn/          g1_vectors.csv, g2_vectors.csv, niche_mlp.pt, scaler.pkl, label_encoder.pkl
                  g1ab_label_consistency.csv, g1ab_label_consistency_cm.png
                  g2_predictions_with_truth.csv, g2_confusion_matrix.csv
                  g2_roc_auc.csv, g2_roc_curves.png, g2_eval_coverage.csv
```